<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">05 Mobile AI</h1>
<hr>


## **1. Understanding Tokenization**


<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>

### 1.1 Introduction

A wireless sensor reports: **“The radio sends a packet.”** We can read and understand the sentence immediately.

A language model does not process words, it first needs a sequence of numbers. 

**Tokenization** is the step that divides text into pieces and looks up an integer identifier for each piece.

A **token** can be a word, part of a word, punctuation, whitespace, or even part of a character's byte representation. A **vocabulary** is the tokenizer's catalogue of pieces; a **token ID** is a piece's index in that catalogue. The ID identifies the piece. It does not describe its meaning.

In this notebook, you will change wireless messages, inspect their pieces and IDs, and watch a small tokenizer learn reusable pieces. You will also see why the same number of words can occupy different amounts of a model's context window. Allow approximately **35–45 minutes**, plus time for the questions.

By the end, you should be able to explain the difference between a word, a token, and a token ID; show what happens to unfamiliar words; and connect tokenization to the embeddings in the next notebook.

#### 1.1.1 Environment Setup

Run the cells from top to bottom. Everything runs on your laptop using Python and the course libraries `matplotlib` and `ipywidgets`. There is no language model to download, and no Raspberry Pi or API key is needed. The tokenizers below are **small teaching examples**, so their IDs and counts are not those of a particular commercial or local LLM.

<div class="alert alert-block alert-success">

If a library is missing, use the installation instructions in [00.1_Check_Course_Environment.ipynb](../00_Intro/00.1_Check_Course_Environment.ipynb). Select that Python environment as this notebook's kernel.
</div>

The sliders and text boxes are for quick experiments. Each experiment also has an ordinary function call: edit its values and rerun the cell to save your own results in the notebook. This also works if your notebook viewer does not display widgets.

In [ ]:
import re
from collections import Counter
from html import escape

import matplotlib.pyplot as plt
from IPython.display import HTML, display

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

print("Ready. All tokenization experiments run locally.")
if widgets is None:
    print("Widgets are unavailable. Edit the function calls and rerun their cells instead.")

### 1.2 Where Would You Cut the Message?

Before running the next experiment, predict how many pieces you would use for `Wi-Fi sends 3 packets.` Should `Wi-Fi` be one piece? What about the hyphen, the number, the full stop, and the spaces?

We compare three deliberately simple rules:

| Rule | What becomes one piece? | What to notice |
|:---|:---|:---|
| Word-like | A run of letters/digits/underscores, a run of whitespace, or one punctuation mark | Easy to read, but a new word needs a new vocabulary entry. |
| Character | One Unicode code point | Small pieces produce a longer sequence. A visible symbol can contain several code points. |
| UTF-8 byte | One byte of the UTF-8 encoded text | There are only 256 possible byte values, but a character may need several bytes. |

In the colored boxes, `·` represents a space, `↵` a newline, and `⇥` a tab. These marks are **display aids**; they are not inserted into your message. A box showing `0x…` contains bytes that cannot be displayed as a complete UTF-8 character on their own.

The next cell contains the display helpers. Run it once; you do not need to edit it.

In [ ]:
# This helper cell can run even if the earlier setup cell was skipped.
import re
from html import escape
from IPython.display import HTML, display

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None


def word_chunks(text):
    """Preserve every character, including whitespace and punctuation."""
    return re.findall(r"\s+|\w+|[^\w\s]", text, flags=re.UNICODE)


def visible(piece):
    if isinstance(piece, bytes):
        try:
            piece = piece.decode("utf-8")
        except UnicodeDecodeError:
            return "0x" + piece.hex(" ")
    return str(piece).replace(" ", "·").replace("\n", "↵").replace("\t", "⇥")


def token_boxes(pieces, ids=None, limit=160):
    colors = ["#deedf4", "#ffe8af", "#dcefdc", "#eedff4", "#f8ded6"]
    boxes = []
    for index, piece in enumerate(pieces[:limit]):
        label = escape(visible(piece))
        identifier = "" if ids is None else f"<br><small>ID {ids[index]}</small>"
        boxes.append(
            f'<span style="display:inline-block;vertical-align:top;margin:3px;'
            f'padding:6px 8px;border:1px solid #647781;border-radius:5px;'
            f'background:{colors[index % len(colors)]};color:#17252d;'
            f'font-family:monospace;overflow-wrap:anywhere;max-width:100%">'
            f'{label}{identifier}</span>'
        )
    if not pieces:
        boxes.append('<em>Empty text: no tokens.</em>')
    if len(pieces) > limit:
        boxes.append(f'<p>Showing the first {limit} of {len(pieces)} pieces.</p>')
    display(HTML('<div style="line-height:1.8">' + ''.join(boxes) + '</div>'))


def table(headers, rows):
    def cell(value, tag):
        return f'<{tag} style="padding:5px 10px;border-bottom:1px solid #bbb;text-align:left;white-space:pre-wrap">{escape(str(value))}</{tag}>'
    head = '<tr>' + ''.join(cell(value, 'th') for value in headers) + '</tr>'
    body = ''.join('<tr>' + ''.join(cell(value, 'td') for value in row) + '</tr>' for row in rows)
    display(HTML('<div style="overflow-x:auto"><table>' + head + body + '</table></div>'))


def live_controls(function, controls):
    """Keep the ordinary cell output above as a saved, static example."""
    if widgets is not None:
        panel = widgets.interactive_output(function, controls)
        display(widgets.VBox(list(controls.values()) + [panel]))


def inspect_cuts(text="Wi-Fi sends 3 packets.", rule="Word-like"):
    if rule == "Word-like":
        pieces = word_chunks(text)
    elif rule == "Character":
        pieces = list(text)
    else:
        pieces = [bytes([value]) for value in text.encode("utf-8")]
    print(f"{rule}: {len(pieces)} pieces | {len(text)} code points | {len(text.encode('utf-8'))} UTF-8 bytes")
    token_boxes(pieces)
    return pieces

In [ ]:
# Edit these two values and rerun to save a different example.
TEXT = "Wi-Fi sends 3 packets."
RULE = "Word-like"  # Try "Character" or "UTF-8 byte".
_ = inspect_cuts(TEXT, RULE)

if widgets is not None:
    live_controls(inspect_cuts, {
        "text": widgets.Textarea(value=TEXT, description="Message:", layout=widgets.Layout(width="95%")),
        "rule": widgets.Dropdown(options=["Word-like", "Character", "UTF-8 byte"], value=RULE, description="Rule:"),
    })

**Try it:** Replace the message with `The Wi-Fi sensor in Zürich reports 20 °C. 📡`. Compare all three rules. Then try two spaces between `Wi-Fi` and `sensor`. A token count is meaningful only when we know which tokenizer produced it.

### 1.3 From Pieces to IDs: A Small Radio Vocabulary

Now give each word-like piece an ID. The following eight sentences are our **training corpus**: the text from which we build this small vocabulary. Here, “training the tokenizer” means constructing its catalogue. We are not training a language model or learning the meaning of the sentences.

The vocabulary is built once and then held fixed. ID `0` is reserved for `<UNK>`, our unknown-piece marker. The other IDs come from sorting the distinct pieces. Choosing a different ordering would assign different numbers without changing the text.

In [ ]:
RADIO_CORPUS = [
    "The radio sends a packet.",
    "The radio receives a packet.",
    "The sensor sends a signal.",
    "The sensor receives a signal.",
    "The wireless sensor sends packets.",
    "The wireless network carries packets.",
    "Wi-Fi shares a channel.",
    "Interference can delay a packet.",
]

word_vocabulary = ["<UNK>"] + sorted({piece for line in RADIO_CORPUS for piece in word_chunks(line)})
word_to_id = {piece: index for index, piece in enumerate(word_vocabulary)}


def encode_words(text):
    return [word_to_id.get(piece, 0) for piece in word_chunks(text)]


def decode_words(ids):
    return ''.join(word_vocabulary[index] for index in ids)


def inspect_words(text):
    pieces = word_chunks(text)
    ids = encode_words(text)
    print("Original:", repr(text))
    token_boxes(pieces, ids)
    restored = decode_words(ids)
    print("Decoded: ", repr(restored))
    print("Exact reconstruction:", restored == text)
    unknown = [piece for piece, index in zip(pieces, ids) if index == 0]
    print("Unknown pieces:", unknown or "none")
    return ids


print("Our training sentences:")
for sentence in RADIO_CORPUS:
    print(" •", sentence)
table(["Token ID", "Stored piece"], [(i, visible(piece)) for i, piece in enumerate(word_vocabulary)])
_ = inspect_words("The radio sends a packet.")

#### 1.3.1 What Happens to a New Word?

The corpus contains `radio`, but not `router`. Predict the result below before running it. Then try `The antenna sends a packet.` or change `radio` to `Radio`.

When a piece becomes `<UNK>`, decoding cannot recover its original spelling. Several different unfamiliar words share the same ID. Growing a whole-word vocabulary helps, but names, new technical terms, spelling variants, and inflections keep creating more possibilities.

Do not rebuild the vocabulary for each test message. A trained model expects the same ID-to-piece mapping that it used during training. Even if another tokenizer emits the same number of tokens, its IDs are not automatically compatible with that model.

In [ ]:
TEST_MESSAGE = "The router sends a packet."
_ = inspect_words(TEST_MESSAGE)

if widgets is not None:
    live_controls(inspect_words, {
        "text": widgets.Textarea(value=TEST_MESSAGE, description="Message:", layout=widgets.Layout(width="95%")),
    })

### 1.4 Learning Reusable Pieces

Instead of keeping a separate entry for every word, a **subword tokenizer** can build unfamiliar words from smaller pieces. Those pieces are learned from repeated patterns; they need not follow dictionary syllables or have meaning by themselves. Subword segmentation is an established way to handle rare words ([Sennrich, Haddow, and Birch, 2016](https://aclanthology.org/P16-1162/)).

We will use a small **byte-pair encoding (BPE)** demonstration:

1. Begin with the 256 possible single bytes as vocabulary entries.
2. Count adjacent pairs of current tokens in the training text.
3. Merge the most frequent pair and add its combined bytes as one new token.
4. Repeat, keeping the learned merge rules in order.

When encoding a new message, apply those **fixed rules** rather than learning new ones. This distinction between learning rules and applying rules is central to BPE ([Hugging Face BPE lesson](https://huggingface.co/learn/llm-course/en/chapter6/5)).

Our version first divides each sentence into the word-like chunks from Section 1.2. It merges only *inside* a chunk, so it cannot merge a word with its neighboring space or punctuation. This makes the demonstration easy to inspect. Production tokenizers have their own splitting, normalization, and special-token rules; this demonstration does not reproduce any one model's tokenizer.

Because we include **all 256 bytes**, any ordinary Unicode message can be encoded as UTF-8, even if its words or characters never appeared in the corpus. To recover the text, we concatenate all token bytes before decoding. A token by itself may contain only part of a character.

The next cell implements these rules. You can focus on the experiments below; expanding the cell reveals the complete algorithm.

In [ ]:
from collections import Counter


def merge_pair(sequence, pair, new_id):
    """Replace non-overlapping occurrences from left to right."""
    merged = []
    position = 0
    while position < len(sequence):
        if position + 1 < len(sequence) and tuple(sequence[position:position + 2]) == pair:
            merged.append(new_id)
            position += 2
        else:
            merged.append(sequence[position])
            position += 1
    return merged


def train_bpe(corpus, max_merges=80):
    chunks = [list(piece.encode("utf-8")) for line in corpus for piece in word_chunks(line)]
    vocabulary = {index: bytes([index]) for index in range(256)}
    rules = []
    history = []
    for step in range(max_merges):
        counts = Counter(pair for chunk in chunks for pair in zip(chunk, chunk[1:]))
        if not counts:
            break
        # Highest count wins; numeric ID order resolves ties reproducibly.
        ranked = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
        pair, frequency = ranked[0]
        if frequency < 2:
            break
        new_id = 256 + len(rules)
        vocabulary[new_id] = vocabulary[pair[0]] + vocabulary[pair[1]]
        before = sum(map(len, chunks))
        chunks = [merge_pair(chunk, pair, new_id) for chunk in chunks]
        rules.append((pair, new_id))
        history.append({
            "pair": pair, "new_id": new_id, "frequency": frequency,
            "top_pairs": ranked[:5], "before": before, "after": sum(map(len, chunks)),
        })
    return {"vocabulary": vocabulary, "rules": rules, "history": history}


def encode_bpe(text, model, steps=None):
    rules = model["rules"] if steps is None else model["rules"][:steps]
    chunks = [list(piece.encode("utf-8")) for piece in word_chunks(text)]
    for pair, new_id in rules:
        chunks = [merge_pair(chunk, pair, new_id) for chunk in chunks]
    return [index for chunk in chunks for index in chunk]


def decode_bpe(ids, model):
    return b''.join(model["vocabulary"][index] for index in ids).decode("utf-8")


BPE_CORPUS = RADIO_CORPUS + [
    "A wireless radio receives wireless packets.",
    "A wireless network connects wireless sensors.",
    "The radio channel carries a signal.",
    "The radio channel carries signals.",
    "A packet arrives after another packet.",
    "Retransmission sends the packet again.",
    "Interference affects the wireless channel.",
    "Interference affects the radio signal.",
    "The sensor uses a battery.",
    "The network connects a radio and a sensor.",
]
BPE_MODEL = train_bpe(BPE_CORPUS)
N_MERGES = len(BPE_MODEL["rules"])
print(f"Learned {N_MERGES} merge rules from {len(BPE_CORPUS)} wireless sentences.")
print(f"Vocabulary: 256 single-byte entries + {N_MERGES} merged entries.")

#### 1.4.1 Watch the Merges

At step `0`, each byte is a separate token. Move the slider one step at a time. The table shows which pair was most frequent in the **whole training corpus** just before that step. The boxes show how the selected rules affect your test message.

Some steps will leave your message unchanged: that pair occurred elsewhere in the corpus. Watch `wireless`, `radio`, and `packet` at different steps. Their pieces are consequences of the corpus, not hand-written wireless vocabulary rules.

In [ ]:
def inspect_merges(step=0, text="The wireless radio sends a packet."):
    step = max(0, min(int(step), N_MERGES))
    print(f"Merge rules applied: {step}/{N_MERGES} | Vocabulary entries available: {256 + step}")
    if step:
        event = BPE_MODEL["history"][step - 1]
        vocabulary = BPE_MODEL["vocabulary"]
        table(["Candidate pair before this merge", "Adjacent occurrences"], [
            (visible(vocabulary[a]) + " + " + visible(vocabulary[b]), frequency)
            for (a, b), frequency in event["top_pairs"]
        ])
        print("Selected:", visible(vocabulary[event["new_id"]]), "-> token ID", event["new_id"])
        print(f"Training-corpus tokens: {event['before']} -> {event['after']}")
    ids = encode_bpe(text, BPE_MODEL, step)
    token_boxes([BPE_MODEL["vocabulary"][i] for i in ids], ids)
    print("Message tokens:", len(ids), "| Exact reconstruction:", decode_bpe(ids, BPE_MODEL) == text)
    return ids


MERGE_STEP = min(24, N_MERGES)
MERGE_MESSAGE = "The wireless radio sends a packet."
_ = inspect_merges(MERGE_STEP, MERGE_MESSAGE)

if widgets is not None:
    live_controls(inspect_merges, {
        "step": widgets.IntSlider(value=MERGE_STEP, min=0, max=N_MERGES, description="Merges:", continuous_update=False),
        "text": widgets.Textarea(value=MERGE_MESSAGE, description="Message:", layout=widgets.Layout(width="95%")),
    })

#### 1.4.2 What Do We Gain, and What Do We Pay?

The plots below track this experiment from zero merges to the final rules. More entries give the tokenizer more opportunities to represent repeated byte sequences with a single ID. Our catalogue grows while the training-corpus sequence gets shorter. A particular message may stay unchanged at some steps.

Fewer tokens do not automatically mean a better language model. A larger vocabulary also needs more embedding rows, which you will meet in the next notebook. Nor is the token count a compressed-file size: storing IDs and a vocabulary also costs bits. Here we are choosing model input units, not measuring a file compressor.

In [ ]:
import matplotlib.pyplot as plt

steps = list(range(N_MERGES + 1))
training_counts = [sum(len(line.encode("utf-8")) for line in BPE_CORPUS)]
training_counts += [event["after"] for event in BPE_MODEL["history"]]
message_counts = [len(encode_bpe(MERGE_MESSAGE, BPE_MODEL, step)) for step in steps]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].plot(steps, [256 + step for step in steps], color="#246b8e")
axes[0].set(xlabel="Learned merge rules", ylabel="Vocabulary entries", title="A growing catalogue")
axes[1].plot(steps, training_counts, color="#246b8e")
axes[1].set(xlabel="Learned merge rules", ylabel="Number of tokens", title="Training corpus")
axes[2].plot(steps, message_counts, color="#a84d13")
axes[2].set(xlabel="Learned merge rules", ylabel="Number of tokens", title="Selected message")
for axis in axes:
    axis.grid(alpha=0.2)
fig.tight_layout()
plt.show()
print("Selected message:", repr(MERGE_MESSAGE))
print("Message token count, before and after:", message_counts[0], "->", message_counts[-1])

### 1.5 Unfamiliar Words, Spaces, and Unicode

Compare `radio` with `radios`, change a letter's case, or use a place name outside the corpus. The byte-based tokenizer can preserve the text even when its pieces are small. **Being representable does not mean being understood:** vocabulary coverage tells us nothing by itself about a model's knowledge of the message.

The next table tests exact reconstruction, including whitespace. This teaching tokenizer does not lowercase, normalize, or discard characters. Other tokenizers can make different choices, so exact reconstruction is a property to check rather than assume for every tokenizer ([Hugging Face tokenizer overview](https://huggingface.co/docs/transformers/tokenizer_summary)).

In [ ]:
WIRELESS_EXAMPLES = [
    "The radio sends a packet.",
    "The radios send a packet.",
    "Wi-Fi uses channel 6.",
    "wi-fi uses channel 6.",
    "Wi-Fi  uses channel 6.",
    "The sensor in Zürich reports 20 °C. 📡",
    "RSSI=-67 dBm\nSNR=18 dB",
]
rows = []
for message in WIRELESS_EXAMPLES:
    ids = encode_bpe(message, BPE_MODEL)
    rows.append((repr(message), len(message), len(message.encode("utf-8")), len(ids), decode_bpe(ids, BPE_MODEL) == message))
table(["Message", "Code points", "UTF-8 bytes", "BPE tokens", "Exact reconstruction"], rows)

UNICODE_MESSAGE = "Zürich 📡"
unicode_ids = encode_bpe(UNICODE_MESSAGE, BPE_MODEL)
token_boxes([BPE_MODEL["vocabulary"][i] for i in unicode_ids], unicode_ids)
print("Concatenate all token bytes, then decode:", decode_bpe(unicode_ids, BPE_MODEL))

#### 1.5.1 A Token Is Not a Letter

How many `e` letters are in `interference`? Predict the count, then inspect the token boundaries. Counting tokens whose text contains `e` is not the same operation as counting every occurrence of the letter.

This illustrates why a token boundary is not a character boundary. It does **not** establish whether a particular LLM will answer a spelling question correctly, or identify a single cause for any mistake. Here we can recover the text and count the letters exactly with Python.

In [ ]:
WORD = "interference"
LETTER = "e"
word_ids = encode_bpe(WORD, BPE_MODEL)
pieces = [BPE_MODEL["vocabulary"][index] for index in word_ids]
token_boxes(pieces, word_ids)
restored = decode_bpe(word_ids, BPE_MODEL)
print("Tokens:", len(word_ids))
print(f"Tokens containing at least one {LETTER!r}:", sum(LETTER.encode("utf-8") in piece for piece in pieces))
print(f"Occurrences of {LETTER!r} in the decoded word:", restored.count(LETTER))
print("Character positions (starting at 1):", [i + 1 for i, char in enumerate(restored) if char == LETTER])

### 1.6 How Much Fits in a Context Window?

A context window limits how many token positions a model can use. In a simplified text-only conversation, the prompt, any conversation formatting, and the generated reply must fit within that budget. The relevant count comes from the model's own tokenizer, not from the number of words shown on screen.

Our workshop example uses a **128-token context**, reserves **24 tokens for a reply**, and assumes **4 formatting tokens**. These are invented classroom settings, not the specifications of a real model. The remaining room is:

$$
\text{remaining} = \text{context limit} - \text{message tokens} - \text{formatting tokens} - \text{reply reserve}.
$$

Try adding repeated wireless measurements. When the total exceeds the context limit, the display reports how far over it is. It does not silently truncate your message or claim that the model can process it anyway.

In [ ]:
PROMPT = "Explain why interference can delay a packet. The wireless sensor reports a weak signal."
CONTEXT_LIMIT = 128
REPLY_RESERVE = 24
FORMATTING_TOKENS = 4


def inspect_budget(text=PROMPT, context=CONTEXT_LIMIT, reply=REPLY_RESERVE):
    count = len(encode_bpe(text, BPE_MODEL))
    used = count + FORMATTING_TOKENS + reply
    remaining = context - used
    print(f"Message {count} + formatting {FORMATTING_TOKENS} + reply reserve {reply} = {used} / {context} tokens")
    print(f"Room left: {remaining} tokens" if remaining >= 0 else f"Does not fit: {-remaining} tokens over the limit.")
    sections = [("Message", count, "#246b8e"), ("Formatting", FORMATTING_TOKENS, "#75606e"),
                ("Reply reserve", reply, "#a84d13"), ("Room left", max(0, remaining), "#577957")]
    scale = max(context, used, 1)
    bars = ''.join(f'<div title="{label}: {amount}" style="width:{100 * amount / scale:.3f}%;height:28px;background:{color}"></div>'
                   for label, amount, color in sections if amount)
    display(HTML('<div style="display:flex;border:1px solid #555;width:100%;max-width:850px">' + bars + '</div>'))
    table(["Component", "Tokens"], [(label, amount) for label, amount, color in sections])
    print("Teaching tokenizer only. Real limits require the actual model's tokenizer and chat format.")
    return {"message": count, "used": used, "remaining": remaining}


baseline_budget = inspect_budget(PROMPT, CONTEXT_LIMIT, REPLY_RESERVE)
if widgets is not None:
    live_controls(inspect_budget, {
        "text": widgets.Textarea(value=PROMPT, description="Prompt:", layout=widgets.Layout(width="95%", height="85px")),
        "context": widgets.IntSlider(value=CONTEXT_LIMIT, min=32, max=256, step=16, description="Context:", continuous_update=False),
        "reply": widgets.IntSlider(value=REPLY_RESERVE, min=0, max=128, step=8, description="Reply:", continuous_update=False),
    })

For a mobile assistant, the amount of text represented by those tokens matters when deciding how many sensor readings or earlier messages to include. Token count is useful for budgeting; it is not a timing benchmark. Runtime also depends on the model, hardware, and how the computation is performed.

### 1.7 From Token IDs to Embeddings

We have reached the input to the next notebook:

**Wireless text → token pieces → token IDs → embedding vectors → attention and prediction**

For each token ID, a model looks up a row in its embedding table. Those row values are model parameters learned during training. A tokenizer's IDs are catalogue indices: ID `310` is not “more wireless” than ID `27`, and neighboring IDs need not have similar meanings.

In [05.02 Understanding Word Embedding](05.02_Understanding_Word_Embedding.ipynb), you will work with vectors and their relationships. In [05.18 Build Your Own Model](05.18_Build_Your_Own_Model.ipynb), the course uses a deliberately simple word-level tokenizer to keep the model inspectable. You can now explain both the appeal and the limitations of that choice.

### 1.8 Your Wireless Playground

Create a short message that a radio, sensor, access point, or network administrator might produce. Keep the BPE vocabulary fixed and compare the word-level and BPE encodings below.

For a controlled experiment, save your baseline result, change **one thing** in the message, and rerun this cell. Try a new device name, a plural, a changed unit, a newline, or a word in another language. Record the change and its effect in the questions. The smallest token count is not automatically the best result: check whether the text survived decoding.

In [ ]:
# Your experiment: change this message, not the trained vocabularies above.
MY_MESSAGE = "The wireless sensor sends a packet."

print("WORD-LEVEL TOKENIZER")
my_word_ids = inspect_words(MY_MESSAGE)
print("\nBYTE-PAIR TOKENIZER")
my_bpe_ids = encode_bpe(MY_MESSAGE, BPE_MODEL)
token_boxes([BPE_MODEL["vocabulary"][index] for index in my_bpe_ids], my_bpe_ids)
print("BPE token IDs:", my_bpe_ids)
print("Decoded:", repr(decode_bpe(my_bpe_ids, BPE_MODEL)))
print("Exact reconstruction:", decode_bpe(my_bpe_ids, BPE_MODEL) == MY_MESSAGE)
table(["Measurement", "Value"], [
    ("Word-level tokens", len(my_word_ids)),
    ("Unknown word-level pieces", my_word_ids.count(0)),
    ("BPE tokens", len(my_bpe_ids)),
    ("UTF-8 bytes in original text", len(MY_MESSAGE.encode("utf-8"))),
])

### 1.9 Summarizing Remarks

Tokenization makes text addressable by a model. A fixed vocabulary connects pieces to IDs, and the model then connects those IDs to vectors. Whole-word pieces are easy to read, but unfamiliar words expose the limits of a small vocabulary. Reusable subword pieces offer another trade-off; starting from all byte values lets our example preserve text outside its training corpus.

Keep three questions separate when looking at an unfamiliar message: **Can the tokenizer represent it? How many tokens does it need? Does the model understand it?** This notebook lets you measure the first two. The third requires evaluating a language model on a task.

### 1.10 Further Reading and Literature

[01] Sennrich, R., Haddow, B., and Birch, A. (2016). [Neural Machine Translation of Rare Words with Subword Units](https://aclanthology.org/P16-1162/). ACL. Research introducing subword units for rare-word handling in neural machine translation.

[02] Hugging Face. [Byte-Pair Encoding tokenization](https://huggingface.co/learn/llm-course/en/chapter6/5). A worked explanation of training and applying BPE merge rules.

[03] Hugging Face. [Tokenization algorithms](https://huggingface.co/docs/transformers/tokenizer_summary). An overview of word, character, and subword tokenization and differences between tokenizer families.

---
### 1.11 Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

Use your own messages and outputs as evidence. For each experiment, copy the relevant counts, pieces, or IDs into your answer so that the result remains visible after saving.

**Question 01: Pieces, characters, and bytes**

Use one wireless sentence from Section 1.2. Report its word-like piece count, code-point count, and UTF-8 byte count. Identify one boundary or count that surprised you and explain it.

(answer here)

**Question 02: A fixed vocabulary**

In Section 1.3, change `radio` to `router` without rebuilding the vocabulary. Copy the changed pieces, their IDs, and the decoded text. Explain what information was lost and why.

(answer here)

**Question 03: One BPE merge**

Choose a merge step in Section 1.4.1. Report the selected pair, its adjacent-occurrence count, and its new token ID. Show the message before and after that step. If it did not change, explain why a rule can be useful to the training corpus without affecting this message.

(answer here)

**Question 04: Vocabulary versus sequence length**

For the selected message in Section 1.4.2, report vocabulary size and message token count at zero merges and at the final step. Explain the trade-off, including why fewer tokens do not by themselves prove that a model is better or a file is smaller.

(answer here)

**Question 05: An unfamiliar wireless message**

In Section 1.8, use a device name or place name that is absent from the training corpus. Report the word-level unknown-piece count, the BPE pieces and IDs, and whether each tokenizer reconstructs the message. Explain what this tells you about representation and what it does not tell you about understanding.

(answer here)

**Question 06: A context budget**

Use Section 1.6 to create a wireless prompt that does not fit. Report message tokens, formatting tokens, reply reserve, context limit, and the amount over the limit. Make one change that makes it fit and report the new numbers. Explain any information or reply space you gave up.

(answer here)

**Question 07: Tokens and letters**

In Section 1.5.1, report the BPE pieces for `interference` and the positions of its `e` letters. Explain why counting token boxes is not a method for counting letters.

(answer here)

**Question 08: The next notebook**

Choose two BPE token IDs from your Section 1.8 output. Explain why comparing their numerical values does not measure semantic similarity, and describe what the embedding lookup will do with each ID.

(answer here)

**Optional feedback**

Which explanation or experiment should be improved? Describe what was unclear or suggest a wireless example that would help the next class.

(answer here)